# 统计和建模 / Statistics and Modeling

---

统计学是数学领域的一个重要分支，几乎涉及科学和工程学的所有应用领域，也被广泛应用于商业、金融等需要依靠数据来获取知识和做出决策的领域。

Statistics is an important branch of mathematics that touches almost every applied field in science and engineering, and is also widely used in business, finance, and other areas where data-driven knowledge and decisions are needed.

本章将介绍使SciPy中的stats模块，包括描述性统计量、随机数、随机变量、分布以及假设检验等内容。

This chapter introduces the `stats` module of SciPy, covering descriptive statistics, random numbers, random variables, distributions, and hypothesis testing.

<!-- bilingual -->

## 导入模块 / Importing Modules

---

<!-- bilingual -->

In [ ]:
import numpy as np
import random

from scipy import stats
from scipy import optimize

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

## 描述性统计量 / Descriptive Statistics

---

<!-- bilingual -->

In [ ]:
x = np.array([3.5, 1.1, 3.2, 2.8, 6.7, 4.4, 0.9, 2.2])

In [ ]:
np.mean(x)

In [ ]:
np.median(x)

In [ ]:
x.min(), x.max()

In [ ]:
x.var()

In [ ]:
x.std()

参数ddof用于设置自由度的个数。如果计算方差的无偏估计以及样本的标准差，需要把ddof设置为1 。

The `ddof` parameter sets the number of degrees of freedom. If the unbiased estimator of variance or the sample standard deviation is desired, set `ddof` to 1.

<!-- bilingual -->

In [ ]:
x.var(ddof=1)

In [ ]:
x.std(ddof=1)

## 随机数 / Random Numbers

---

<!-- bilingual -->

In [ ]:
random.seed(123456789)

In [ ]:
random.random()

In [ ]:
random.randint(0, 10)  # 0 and 10 inclusive

In [ ]:
np.random.seed(123456789)

In [ ]:
np.random.rand()

In [ ]:
np.random.randn()

In [ ]:
np.random.rand(5)

In [ ]:
np.random.randn(2, 4)

In [ ]:
np.random.randint(10, size=10)

In [ ]:
np.random.randint(low=10, high=20, size=(2, 10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].hist(np.random.rand(10000))
axes[0].set_title("rand")
axes[1].hist(np.random.randn(10000))
axes[1].set_title("randn")
axes[2].hist(np.random.randint(low=1, high=10, size=10000), bins=9, align='left')
axes[2].set_title("randint(low=1, high=10)")

fig.tight_layout()
fig.savefig("/tmp/ch13-random-hist.pdf")

In [ ]:
#random.sample(range(10), 5)

In [ ]:
np.random.choice(10, 5, replace=False)

In [ ]:
np.random.seed(123456789)

In [ ]:
np.random.rand()

In [ ]:
np.random.seed(123456789); np.random.rand()

In [ ]:
np.random.seed(123456789); np.random.rand()

In [ ]:
prng = np.random.RandomState(123456789)

In [ ]:
prng.rand(2, 4)

In [ ]:
prng.chisquare(1, size=(2, 2))

In [ ]:
prng.standard_t(1, size=(2, 3))

In [ ]:
prng.f(5, 2, size=(2, 4))

In [ ]:
prng.binomial(10, 0.5, size=10)

In [ ]:
prng.poisson(5, size=10)

## 随机变量及其分布 / Random Variables and Their Distributions

---

<!-- bilingual -->

In [ ]:
np.random.seed(123456789)

In [ ]:
X = stats.norm(1, 0.5)

In [ ]:
X.mean()

In [ ]:
X.median()

In [ ]:
X.std()

In [ ]:
X.var()

In [ ]:
[X.moment(n) for n in range(5)]

In [ ]:
X.stats()

In [ ]:
X.pdf([0, 1, 2])

In [ ]:
X.cdf([0, 1, 2])

In [ ]:
X.rvs(10)

In [ ]:
stats.norm(1, 0.5).stats()

In [ ]:
stats.norm.stats(loc=2, scale=0.5)

In [ ]:
X.interval(0.95)

In [ ]:
X.interval(0.99)

In [ ]:
def plot_rv_distribution(X, axes=None):
    """Plot the PDF, CDF, SF and PPF of a given random variable"""
    if axes is None:
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    
    x_min_999, x_max_999 = X.interval(0.999)
    x999 = np.linspace(x_min_999, x_max_999, 1000)

    x_min_95, x_max_95 = X.interval(0.95)
    x95 = np.linspace(x_min_95, x_max_95, 1000)

    if hasattr(X.dist, 'pdf'):
        axes[0].plot(x999, X.pdf(x999), label="PDF")
        axes[0].fill_between(x95, X.pdf(x95), alpha=0.25)
    else:
        x999_int = np.unique(x999.astype(int))
        axes[0].bar(x999_int, X.pmf(x999_int), label="PMF")
    axes[1].plot(x999, X.cdf(x999), label="CDF")
    axes[1].plot(x999, X.sf(x999), label="SF")
    axes[2].plot(x999, X.ppf(x999), label="PPF")
    
    for ax in axes:
        ax.legend()
    
    return axes

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))

X = stats.norm()
plot_rv_distribution(X, axes=axes[0, :])
axes[0, 0].set_ylabel("Normal dist.")
X = stats.f(2, 50)
plot_rv_distribution(X, axes=axes[1, :])
axes[1, 0].set_ylabel("F dist.")
X = stats.poisson(5)
plot_rv_distribution(X, axes=axes[2, :])
axes[2, 0].set_ylabel("Poisson dist.")

fig.tight_layout()

In [ ]:
def plot_dist_samples(X, X_samples, title=None, ax=None):
    """ Plot the PDF and histogram of samples of a continuous random variable"""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    x_lim = X.interval(.99)
    x = np.linspace(*x_lim, num=100)

    ax.plot(x, X.pdf(x), label="PDF", lw=3)    
    ax.hist(X_samples, label="samples", bins=75)
    ax.set_xlim(*x_lim)
    ax.legend()
    
    if title:
        ax.set_title(title)
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
X = stats.t(7.0)
plot_dist_samples(X, X.rvs(2000), "Student's t dist.", ax=axes[0])
X = stats.chi2(5.0)
plot_dist_samples(X, X.rvs(2000), r"$\chi^2$ dist.", ax=axes[1])
X = stats.expon(0.5)
plot_dist_samples(X, X.rvs(2000), "exponential dist.", ax=axes[2])
fig.tight_layout()

In [ ]:
X = stats.chi2(df=5)

In [ ]:
X_samples = X.rvs(500)

In [ ]:
df, loc, scale = stats.chi2.fit(X_samples)

In [ ]:
df, loc, scale

In [ ]:
Y = stats.chi2(df=df, loc=loc, scale=scale)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 3))

x_lim = X.interval(.99)
x = np.linspace(*x_lim, num=100)

ax.plot(x, X.pdf(x), label="original")
ax.plot(x, Y.pdf(x), label="recreated")
ax.legend()

fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_lim = X.interval(.99)
x = np.linspace(*x_lim, num=100)

axes[0].plot(x, X.pdf(x), label="original")
axes[0].plot(x, Y.pdf(x), label="recreated")
axes[0].legend()

axes[1].plot(x, X.pdf(x) - Y.pdf(x), label="error")
axes[1].legend()

fig.tight_layout()

## 假设检验 / Hypothesis Testing

---

<!-- bilingual -->

In [ ]:
np.random.seed(123456789)

In [ ]:
mu, sigma = 1.0, 0.5

In [ ]:
X = stats.norm(mu-0.2, sigma)

In [ ]:
n = 100

In [ ]:
X_samples = X.rvs(n)

In [ ]:
z = (X_samples.mean() - mu)/(sigma/np.sqrt(n))

In [ ]:
z

In [ ]:
t = (X_samples.mean() - mu)/(X_samples.std(ddof=1)/np.sqrt(n))

In [ ]:
t

In [ ]:
stats.norm().ppf(0.025)

In [ ]:
2 * stats.norm().cdf(-abs(z))

In [ ]:
2 * stats.t(df=(n-1)).cdf(-abs(t))

In [ ]:
t, p = stats.ttest_1samp(X_samples, mu)

In [ ]:
t

In [ ]:
p

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

sns.distplot(X_samples, ax=ax)
x = np.linspace(*X.interval(0.999), num=100)
ax.plot(x, stats.norm(loc=mu, scale=sigma).pdf(x))

fig.tight_layout()

In [ ]:
n = 50

In [ ]:
mu1, mu2 = np.random.rand(2)

In [ ]:
X1 = stats.norm(mu1, sigma)

In [ ]:
X1_sample = X1.rvs(n)

In [ ]:
X2 = stats.norm(mu2, sigma)

In [ ]:
X2_sample = X2.rvs(n)

In [ ]:
t, p = stats.ttest_ind(X1_sample, X2_sample)

In [ ]:
t

In [ ]:
p

In [ ]:
mu1, mu2

In [ ]:
sns.histplot(X1_sample)
sns.histplot(X2_sample)

## 非参数法 / Nonparametric Methods

---

<!-- bilingual -->

In [ ]:
np.random.seed(0)

In [ ]:
X = stats.chi2(df=5)

In [ ]:
X_samples = X.rvs(100)

In [ ]:
kde = stats.kde.gaussian_kde(X_samples)

In [ ]:
kde_low_bw = stats.kde.gaussian_kde(X_samples, bw_method=0.25)

In [ ]:
x = np.linspace(0, 20, 100)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].hist(X_samples, alpha=0.5, bins=25)
axes[1].plot(x, kde(x), label="KDE")
axes[1].plot(x, kde_low_bw(x), label="KDE (low bw)")
axes[1].plot(x, X.pdf(x), label="True PDF")
axes[1].legend()
sns.histplot(X_samples, bins=25, ax=axes[2])

fig.tight_layout()

In [ ]:
kde.resample(10)

In [ ]:
def _kde_cdf(x):
    return kde.integrate_box_1d(-np.inf, x)

In [ ]:
kde_cdf = np.vectorize(_kde_cdf)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 3))

sns.histplot(X_samples, bins=25, ax=ax)
x = np.linspace(0, 20, 100)
ax.plot(x, kde_cdf(x))

fig.tight_layout()

In [ ]:
def _kde_ppf(q):
    return optimize.fsolve(lambda x, q: kde_cdf(x) - q, kde.dataset.mean(), args=(q,))[0]

In [ ]:
kde_ppf = np.vectorize(_kde_ppf)

In [ ]:
kde_ppf([0.05, 0.95])

In [ ]:
X.ppf([0.05, 0.95])